# Lab 10 — Funding Carry on Hyperliquid

**COS/ECE 473 · Elements of Tokenized Finance**

### Goal

You run a **$10,000 market-neutral carry book**. Your task is not merely to find the highest funding rate. You must decide:

1. **Which markets are investable?**
2. **Which funding rates look persistent rather than transient?**
3. **Which trade would you put on, and for how long?**
4. **Does the carry survive fees and financing?**
5. **What historically motivated stress would make you change your mind?**

This is a paper-trading lab. The notebook only reads public market data; it never signs or submits an order.

**Expected time:** 30–45 minutes.


## What is provided vs. what you decide

The data plumbing and accounting formulas are provided. Your work is concentrated in the **decision variables**.

You will make four substantive choices:

- **TODO 1 — Investability:** choose variables and thresholds beyond the mandatory $1M 24h-volume screen.
- **TODO 2 — Persistence:** choose historical funding statistics that tell you whether current carry is likely to persist.
- **TODO 3 — Trade construction:** choose the market, expected-funding estimator, and holding period.
- **TODO 4 — Stress:** choose a downside scenario motivated by observed history.

Finally, you make the decision whether to place the trade or not (**TODO 5**)

There is deliberately **no unique correct answer**. Your rule should be internally coherent and economically defensible.


## Verified Hyperliquid conventions used in this lab

Public Info API: `POST https://api.hyperliquid.xyz/info`

- `metaAndAssetCtxs` returns the perpetual universe and contexts including current `funding`, `markPx`, `oraclePx`, `openInterest`, and `dayNtlVlm`.
- `spotMetaAndAssetCtxs` returns spot metadata, which we use to identify underlyings with a Hyperliquid USDC spot market.
- `fundingHistory` returns historical hourly `fundingRate` observations for a coin.
- Hyperliquid funding is paid **hourly**. Positive funding means **longs pay shorts**.
- Base Tier-0 taker fees in the current fee schedule are **0.045% for perps** and **0.070% for spot**.

For a long-spot / short-perp carry trade executed and closed with taker orders, the standardized four-fill fee hurdle is

2 x (7+4.5) = 23bps.


Sources:
- https://hyperliquid.gitbook.io/hyperliquid-docs/for-developers/api/info-endpoint/perpetuals
- https://hyperliquid.gitbook.io/hyperliquid-docs/for-developers/api/info-endpoint/spot
- https://hyperliquid.gitbook.io/hyperliquid-docs/trading/funding
- https://hyperliquid.gitbook.io/hyperliquid-docs/trading/fees


In [ ]:
import math
import time

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

INFO_URL = "https://api.hyperliquid.xyz/info"

# Keep False in class. True uses a deterministic offline fixture for debugging.
USE_FIXTURE = False

HOURS_PER_YEAR = 24 * 365
NOTIONAL_USD = 10_000

# Current base Tier-0 taker fees from Hyperliquid docs.
PERP_TAKER_BPS = 4.5
SPOT_TAKER_BPS = 7.0

# Illustrative financing / opportunity cost on the fully funded spot leg.
# This is an assumption, not a Hyperliquid protocol parameter.
SPOT_FINANCING_APR = 0.05

# Core risk screen requested for the lab.
MIN_24H_VOLUME_USD = 1_000_000


In [ ]:
def post_info(payload):
    r = requests.post(INFO_URL, json=payload, timeout=20)
    r.raise_for_status()
    return r.json()

def get_hl_spot_bases():
    '''Return base-token names with a Hyperliquid spot pair quoted in USDC.'''
    spot_meta, _ = post_info({"type": "spotMetaAndAssetCtxs"})
    token_name = {int(t["index"]): t["name"] for t in spot_meta["tokens"]}

    bases = set()
    for pair in spot_meta["universe"]:
        base_idx, quote_idx = pair["tokens"]
        if token_name.get(int(quote_idx)) == "USDC":
            bases.add(token_name.get(int(base_idx)))
    return bases

def get_perp_cross_section():
    meta, ctxs = post_info({"type": "metaAndAssetCtxs"})
    universe = meta["universe"]

    if len(universe) != len(ctxs):
        raise ValueError("Hyperliquid universe and asset-context arrays are not aligned.")

    spot_bases = get_hl_spot_bases()
    rows = []

    for asset, ctx in zip(universe, ctxs):
        if asset.get("isDelisted", False):
            continue

        coin = asset["name"]
        oracle = float(ctx["oraclePx"])
        mark = float(ctx["markPx"])
        oi_base = float(ctx["openInterest"])
        volume = float(ctx["dayNtlVlm"])
        funding = float(ctx["funding"])
        oi_notional = oi_base * oracle

        rows.append({
            "coin": coin,
            "funding_hourly": funding,
            "funding_ann_pct": funding * HOURS_PER_YEAR * 100,
            "oracle_px": oracle,
            "mark_px": mark,
            "basis_bps": (mark / oracle - 1) * 10_000,
            "abs_basis_bps": abs(mark / oracle - 1) * 10_000,
            "open_interest_base": oi_base,
            "oi_notional_usd": oi_notional,
            "volume_24h_usd": volume,
            "turnover_24h": volume / oi_notional if oi_notional > 0 else np.nan,
            "max_leverage": asset.get("maxLeverage", np.nan),
            "hl_spot_usdc": coin in spot_bases,
        })

    return pd.DataFrame(rows)

markets = get_perp_cross_section()

base_universe = markets[
    (markets["volume_24h_usd"] >= MIN_24H_VOLUME_USD)
    & (markets["hl_spot_usdc"])
].copy()

SPOT_FEE_IS_BENCHMARK = False

# Robustness guard: if too few same-venue spot/perp pairs exist in a live snapshot,
# keep the lab usable by allowing external spot execution. In that case the 7 bps
# spot fee below is a standardized benchmark rather than an exact venue fee.
if len(base_universe) < 3:
    SPOT_FEE_IS_BENCHMARK = True
    base_universe = markets[
        markets["volume_24h_usd"] >= MIN_24H_VOLUME_USD
    ].copy()
    print(
        "WARNING: fewer than 3 $1M-volume perps also have a Hyperliquid USDC spot pair. "
        "Using all $1M-volume perps; the 7 bps spot fee is now a benchmark for the external spot leg."
    )

print(f"All active perps: {len(markets)}")
print(f"Baseline universe: {len(base_universe)}")
print(f"Spot fee treated as benchmark: {SPOT_FEE_IS_BENCHMARK}")

display(
    base_universe.sort_values("funding_ann_pct", ascending=False)[
        ["coin", "funding_ann_pct", "volume_24h_usd", "oi_notional_usd",
         "abs_basis_bps", "turnover_24h", "max_leverage", "hl_spot_usdc"]
    ].head(15)
)


## Part 1 — Build an investable universe

The notebook already imposes:

- **24h notional volume ≥ $1M**
- a **Hyperliquid USDC spot market when at least three such markets are available**. If not, the notebook falls back to >$1M-volume perps and clearly labels the 7 bps spot fee as a standardized external-spot benchmark.

Now add your own risk controls.

### TODO 1

Choose **at least two** additional variables from the live table and explain why they matter. Good candidates include:

- `oi_notional_usd`
- `abs_basis_bps`
- `turnover_24h`
- `max_leverage`

Then implement your filter below. Do **not** simply rank the entire universe by current funding.

**Written rationale (1–2 sentences):**  
`YOUR ANSWER HERE`


In [ ]:
def make_investable_universe(df):
    # TODO 1: choose at least TWO economically motivated conditions.
    # Example syntax only:
    # return df[(df["oi_notional_usd"] >= ...) & (df["abs_basis_bps"] <= ...)].copy()
    raise NotImplementedError("TODO 1: define your investability rule")

investable = make_investable_universe(base_universe)

if len(investable) < 3:
    print("Warning: your rule leaves fewer than 3 markets. Decide whether that is intentional.")

display(
    investable.sort_values("funding_ann_pct", ascending=False)[
        ["coin", "funding_ann_pct", "volume_24h_usd", "oi_notional_usd",
         "abs_basis_bps", "turnover_24h", "max_leverage"]
    ]
)


## Part 2 — Is today's funding believable?

Take the **three highest-current-funding** markets that survive your investability rule. We will pull seven days of hourly funding for them.

The code computes several diagnostics, but **you decide which ones matter**.


In [ ]:
def fetch_funding_history(coin, lookback_days=7):
    end_ms = int(time.time() * 1000)
    start_ms = end_ms - lookback_days * 24 * 60 * 60 * 1000

    raw = post_info({
        "type": "fundingHistory",
        "coin": coin,
        "startTime": start_ms,
        "endTime": end_ms,
    })

    h = pd.DataFrame(raw)
    if h.empty:
        return h

    h["fundingRate"] = h["fundingRate"].astype(float)
    h["time"] = pd.to_datetime(h["time"], unit="ms", utc=True)
    return h.sort_values("time").reset_index(drop=True)

def funding_features(h):
    '''Compute diagnostics; students choose which diagnostics matter.'''
    if len(h) < 24:
        raise ValueError("Need at least 24 funding observations for the persistence diagnostics.")

    r = h["fundingRate"].astype(float)
    ann = r * HOURS_PER_YEAR * 100
    rolling_24h = r.rolling(24).sum()

    return pd.Series({
        "mean_ann_pct": ann.mean(),
        "median_ann_pct": ann.median(),
        "positive_hour_share": (r > 0).mean(),
        "p10_ann_pct": ann.quantile(0.10),
        "funding_vol_ann_pct": ann.std(),
        "worst_24h_funding_pct": rolling_24h.min() * 100,
    })


In [ ]:
candidate_coins = (
    investable.sort_values("funding_ann_pct", ascending=False)
    .head(3)["coin"]
    .tolist()
)

if len(candidate_coins) == 0:
    raise ValueError("Your investability rule produced no candidates.")

histories = {coin: fetch_funding_history(coin, lookback_days=7) for coin in candidate_coins}

feature_rows = []
for coin, h in histories.items():
    f = funding_features(h)
    f["coin"] = coin
    feature_rows.append(f)

persistence = pd.DataFrame(feature_rows).set_index("coin")

current = (
    markets.set_index("coin")
    .loc[persistence.index, ["funding_ann_pct"]]
    .rename(columns={"funding_ann_pct": "current_ann_pct"})
)

candidate_table = current.join(persistence)
display(candidate_table)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for coin, h in histories.items():
    ax.plot(
        h["time"],
        h["fundingRate"] * HOURS_PER_YEAR * 100,
        label=coin,
    )
ax.axhline(0, linewidth=1)
ax.set_ylabel("Funding run-rate (simple annualized %)")
ax.set_title("7-day hourly funding history for current-carry candidates")
ax.legend()
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


### TODO 2 — Choose your persistence variables

Choose **two historical statistics** that you think best answer:

> “If I enter this trade today, how much confidence should I have that positive funding persists?”

Examples:

- median funding
- fraction of positive hours
- 10th percentile funding
- funding volatility
- worst rolling 24h funding

Then write a ranking/filtering rule and select **one** market for deeper analysis.

**Written rationale (2–3 sentences):**  
`YOUR ANSWER HERE`


In [ ]:
# TODO 2:
# Use at least TWO columns from candidate_table to choose the market.
# The rule can filter, rank, or create a simple score.
CHOSEN_COIN = None

if CHOSEN_COIN not in candidate_table.index:
    raise ValueError("TODO 2: choose CHOSEN_COIN from candidate_table.index")

print("Chosen market:", CHOSEN_COIN)
display(candidate_table.loc[[CHOSEN_COIN]])


## Part 3 — Price the trade after costs

Trade structure:

- **Long $10,000 spot**
- **Short $10,000 perpetual**
- First-order price delta is approximately zero.
- You receive funding while funding is positive.

Choose an expected-funding estimator and a holding period.

Allowed estimators:

- `"current"`
- `"mean_7d"`
- `"median_7d"`
- `"p10_7d"`

Holding period must be between **1 and 14 days**.


### TODO 3 — Choose estimator and holding period

Make a choice consistent with what you learned about persistence.

**Written rationale (1–2 sentences):**  
`YOUR ANSWER HERE`


In [ ]:
def expected_hourly_from_method(coin_row, persistence_row, method):
    method = method.lower()
    if method == "current":
        return float(coin_row["funding_hourly"])
    if method == "mean_7d":
        return float(persistence_row["mean_ann_pct"]) / 100 / HOURS_PER_YEAR
    if method == "median_7d":
        return float(persistence_row["median_ann_pct"]) / 100 / HOURS_PER_YEAR
    if method == "p10_7d":
        return float(persistence_row["p10_ann_pct"]) / 100 / HOURS_PER_YEAR
    raise ValueError("method must be one of: current, mean_7d, median_7d, p10_7d")

def carry_accounting(notional, expected_hourly, hold_days):
    gross_funding = notional * expected_hourly * 24 * hold_days

    round_trip_fee_bps = 2 * (SPOT_TAKER_BPS + PERP_TAKER_BPS)
    trading_fees = notional * round_trip_fee_bps / 10_000

    financing = notional * SPOT_FINANCING_APR * hold_days / 365
    net_pnl = gross_funding - trading_fees - financing

    daily_net_before_fees = (
        notional * expected_hourly * 24
        - notional * SPOT_FINANCING_APR / 365
    )
    break_even_days = (
        trading_fees / daily_net_before_fees
        if daily_net_before_fees > 0 else np.inf
    )

    return {
        "gross_funding_usd": gross_funding,
        "trading_fees_usd": trading_fees,
        "spot_financing_usd": financing,
        "net_expected_pnl_usd": net_pnl,
        "round_trip_fee_bps": round_trip_fee_bps,
        "fee_break_even_days": break_even_days,
    }

def stress_worst_24h(notional, expected_hourly, hold_days, history):
    # Replace one expected day with the worst observed rolling 24h funding day.
    r = history["fundingRate"].astype(float)
    worst_24h_sum = r.rolling(24).sum().min()

    total_hours = int(24 * hold_days)
    if total_hours <= 24:
        stressed_funding_fraction = worst_24h_sum * (total_hours / 24)
    else:
        stressed_funding_fraction = worst_24h_sum + expected_hourly * (total_hours - 24)

    stressed_funding = notional * stressed_funding_fraction
    trading_fees = notional * 2 * (SPOT_TAKER_BPS + PERP_TAKER_BPS) / 10_000
    financing = notional * SPOT_FINANCING_APR * hold_days / 365

    return {
        "worst_observed_24h_funding_pct": worst_24h_sum * 100,
        "stressed_funding_usd": stressed_funding,
        "stressed_net_pnl_usd": stressed_funding - trading_fees - financing,
    }


## Risk ledger

The lab **reduces** some risks; it does not make the trade risk-free.

| Risk | What we do in the lab | What remains |
|---|---|---|
| Thin liquidity / capacity | Mandatory 24h volume > $1M; you may add OI/turnover filters | Volume is not executable depth; slippage can still matter |
| Funding reversal | Examine 7d history and design a stress | Future regime can be worse than 7d history |
| Basis | Observe mark/oracle basis; you may screen it | Spot and perp can temporarily diverge |
| Fees | Explicit 23 bps round-trip Tier-0 taker hurdle | Your actual tier may differ; slippage is excluded |
| Financing | 5% APR assumption on spot capital | Actual financing/opportunity cost varies |
| Liquidation / margin | Paper trade is equal-notional and assumes adequate margin | A real short perp can still be liquidated during dislocations |
| Venue / oracle / custody | Not modeled | Smart-contract, oracle, operational, and custody risks remain |

**Key principle:** delta-neutral removes the first-order directional bet. It does **not** remove these residual risks.


In [ ]:
# TODO 3
EXPECTED_RATE_METHOD = None
HOLD_DAYS = None

if EXPECTED_RATE_METHOD not in {"current", "mean_7d", "median_7d", "p10_7d"}:
    raise ValueError("TODO 3: choose a valid EXPECTED_RATE_METHOD")
if not isinstance(HOLD_DAYS, int) or not (1 <= HOLD_DAYS <= 14):
    raise ValueError("TODO 3: choose HOLD_DAYS between 1 and 14")

coin_row = markets.set_index("coin").loc[CHOSEN_COIN]
pers_row = candidate_table.loc[CHOSEN_COIN]

expected_hourly = expected_hourly_from_method(
    coin_row, pers_row, EXPECTED_RATE_METHOD
)

accounting = carry_accounting(
    NOTIONAL_USD, expected_hourly, HOLD_DAYS
)

quantity = NOTIONAL_USD / float(coin_row["oracle_px"])

summary = pd.Series({
    "coin": CHOSEN_COIN,
    "position_notional_usd": NOTIONAL_USD,
    "base_asset_quantity_each_leg": quantity,
    "expected_funding_ann_pct": expected_hourly * HOURS_PER_YEAR * 100,
    "hold_days": HOLD_DAYS,
    **accounting,
})
display(summary.to_frame("value"))


### Sanity check: what did delta-neutral remove?

The following calculation shocks spot and perp by the **same** 20% move. This is stylized — basis can move in reality — but it illustrates first-order delta cancellation.


In [ ]:
P0 = float(coin_row["oracle_px"])
P1 = 1.20 * P0

spot_price_pnl = quantity * (P1 - P0)
short_perp_price_pnl = -quantity * (P1 - P0)

print(f"Long spot price P&L:   ${spot_price_pnl:,.2f}")
print(f"Short perp price P&L: ${short_perp_price_pnl:,.2f}")
print(f"Net price P&L:        ${spot_price_pnl + short_perp_price_pnl:,.2f}")
print("\nFunding, fees, financing, basis, margin, and venue risks remain.")


## Part 4 — Design a stress from the data

Do **not** invent an arbitrary negative funding number.

Use the seven-day history of your chosen market to create a stress scenario. You may use:

- the worst observed rolling 24h funding;
- the 10th-percentile hourly rate for the entire holding period;
- another data-derived scenario you can explain.

A helper for the **worst observed rolling 24h** scenario is provided.


### TODO 4 — Choose and justify the stress

Set `STRESS_METHOD` to `"worst_24h"` or implement your own data-driven stress.

**Written rationale (1–2 sentences):**  
`YOUR ANSWER HERE`


In [ ]:
# TODO 4
STRESS_METHOD = None

chosen_history = histories[CHOSEN_COIN]

if STRESS_METHOD == "worst_24h":
    stress = stress_worst_24h(
        NOTIONAL_USD, expected_hourly, HOLD_DAYS, chosen_history
    )
else:
    raise NotImplementedError(
        "TODO 4: choose 'worst_24h' or implement another historical stress"
    )

display(pd.Series(stress).to_frame("value"))


## Final decision

### TODO 5 — Would you actually put the trade on?

Answer **YES**, **NO**, or **NOT ENOUGH INFORMATION**, and defend the answer in 3–5 sentences.

At minimum, discuss:

- persistence of funding;
- fee break-even time;
- expected net P&L;
- stressed net P&L;
- one residual risk not captured numerically.

`YOUR ANSWER HERE`


## Optional extension — execution depth

24h volume is only a proxy for liquidity. A faster group can replace it with actual executable depth using Hyperliquid's `l2Book` Info endpoint and estimate slippage for a $10,000 order.

This is intentionally **not** part of the core 30–45 minute lab.
